In [4]:
# Environment setup for GPU (pinned, CUDA wheels)
import os
import sys
import subprocess

py = sys.version_info
print(f"Python {py.major}.{py.minor}.{py.micro}")

# Force GPU backend; change to "cpu" only if you want CPU.
os.environ["JAX_PLATFORMS"] = "cuda"

# Clean out any mismatched PJRT plugins.
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "jax-cuda12-plugin", "jax-cuda11-plugin"
], check=False)

# Pinned stack for compatibility with Flax 0.7.x on Python 3.10 (works on 3.11 too).
jax_pkg = "jax==0.4.28"
jaxlib_pkg = "jaxlib==0.4.28"
plugin_pkg = "jax-cuda12-plugin==0.4.28"  # use jax-cuda11-plugin==0.4.28 if your CUDA is 11.x
deps = [
    jax_pkg,
    jaxlib_pkg,
    plugin_pkg,
    "flax==0.7.5",
    "optax==0.2.5",
    "ott-jax==0.6.0",
]

# Torch GPU wheels (CUDA 12.x); adjust index if your CUDA is 11.x
torch_index = "https://download.pytorch.org/whl/cu121"
torch_deps = [
    "torch==2.2.2",
    "torchvision==0.17.2",
]

wheel_index = "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + deps + ["-f", wheel_index])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--extra-index-url", torch_index] + torch_deps)

import jax
print("JAX version:", jax.__version__)
print("Backend:", jax.lib.xla_bridge.get_backend().platform)
print("Devices:", jax.devices())


Python 3.10.12


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.6/143.6 MB 38.9 MB/s  0:00:03:00:0100:01
INFO: pip is looking at multiple versions of lineax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of equinox to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of equinox to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of orbax-checkpoint to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of orbax-checkpoint to determine wh

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtext 0.17.0a0 requires torch==2.2.0a0+81ea7a4, but you have torch 2.2.2+cu121 which is incompatible.


JAX version: 0.4.28
Backend: gpu
Devices: [cuda(id=0)]


In [1]:
"""
Wasserstein-DRO (MNIST) with:
- Algorithm 1: Gradient Descent + Particle Ascent
- Algorithm 2: ICNN-based Gradient Ascent-Descent

Inner maximization:
- Barzilai–Borwein step size + Armijo backtracking
  * particle ascent over x (Algorithm 1)
  * ICNN parameter ascent over ω (Algorithm 2)

ICNN architecture:
- JAX/Flax port of the dense branch of your InputConvexPotential
- Principled non-negative initialization via icnn_principled_moments

Dependencies:
    jax, jaxlib, flax, optax, torch, torchvision
"""

from __future__ import annotations

import os
import csv
os.environ["JAX_PLATFORMS"] = "cuda"  # force GPU backend

import math
import functools
from dataclasses import dataclass
from typing import Any, Dict, Iterable, Optional, Sequence, Tuple

import jax
import jax.numpy as jnp
import jax._src.config as _jax_config_module

# Patch JAX config object with define_* helpers expected by Flax on Python 3.10.
if not hasattr(jax.config, "define_bool_state"):
    for _name in (
        "define_bool_state",
        "define_enum_state",
        "define_float_state",
        "define_int_state",
        "define_optional_enum_state",
        "define_optional_string_state",
        "define_string_or_object_state",
        "define_string_state",
    ):
        if hasattr(_jax_config_module, _name):
            setattr(jax.config, _name, getattr(_jax_config_module, _name))
import optax
import torch
import torchvision

from flax import linen as nn
from flax.training import train_state

from jax.flatten_util import ravel_pytree


class CSVLogger:
    def __init__(self, path: str, fieldnames):
        self.path = path
        self.fieldnames = list(fieldnames)
        directory = os.path.dirname(path)
        if directory:
            os.makedirs(directory, exist_ok=True)
        if not os.path.exists(self.path):
            with open(self.path, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=self.fieldnames)
                writer.writeheader()

    def log(self, **kwargs):
        row = {name: kwargs.get(name) for name in self.fieldnames}
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writerow(row)


DEFAULT_LOG_PATH = os.path.join("MNIST", "training_logs.csv")
LOG_FIELDNAMES = [
    "algorithm",
    "phase",
    "epoch",
    "step",
    "loss_adv",
    "adv_loss",
    "cls_loss",
    "acc_clean",
    "acc_adv",
    "w2_proxy",
]

# ---------------------------------------------------------------------------
#  Basic utilities: loss, accuracy, dataset as arrays
# ---------------------------------------------------------------------------

def cross_entropy_loss(logits: jnp.ndarray, labels: jnp.ndarray) -> jnp.ndarray:
    """Per-example cross-entropy (no mean)."""
    num_classes = logits.shape[-1]
    one_hot = jax.nn.one_hot(labels, num_classes)
    log_probs = jax.nn.log_softmax(logits)
    return -jnp.sum(one_hot * log_probs, axis=-1)


def accuracy(logits: jnp.ndarray, labels: jnp.ndarray) -> jnp.ndarray:
    preds = jnp.argmax(logits, axis=-1)
    return jnp.mean(preds == labels)



def adversary_loss(
    logits: jnp.ndarray,
    labels: jnp.ndarray,
    use_margin_adv: bool,
) -> jnp.ndarray:
    """Adversary loss: cross-entropy or margin log-sum-exp."""
    if use_margin_adv:
        logits_correct = jnp.take_along_axis(logits, labels[:, None], axis=1).squeeze(axis=1)
        margins = logits - logits_correct[:, None]
        num_classes = logits.shape[1]
        mask = jax.nn.one_hot(labels, num_classes=num_classes).astype(bool)
        margins = jnp.where(mask, -jnp.inf, margins)
        return jax.nn.logsumexp(margins, axis=1).mean()

    return cross_entropy_loss(logits, labels).mean()



def pgd_l2_attack(
    apply_fn,
    params,
    x: jnp.ndarray,
    y: jnp.ndarray,
    eps: float,
    num_steps: int,
    step_size: Optional[float] = None,
) -> jnp.ndarray:
    """PGD with L2 projection and grad-normalized adaptive steps."""
    if step_size is None:
        step_size = float(eps) / float(max(num_steps, 1))
    elif step_size <= 0:
        raise ValueError("pgd_l2_attack step_size must be positive.")

    def loss_fn(x_adv):
        logits = apply_fn({"params": params}, x_adv)
        return cross_entropy_loss(logits, y).mean()

    def project_l2(adv):
        diff = adv - x
        flat = diff.reshape((diff.shape[0], -1))
        norm = jnp.linalg.norm(flat, axis=1, keepdims=True)
        factor = jnp.minimum(1.0, eps / (norm + 1e-12))
        factor = factor.reshape((diff.shape[0],) + (1,) * (diff.ndim - 1))
        return x + diff * factor

    def step_fn(_, adv):
        g = jax.grad(loss_fn)(adv)
        g_flat = g.reshape((g.shape[0], -1))
        g_norm = jnp.linalg.norm(g_flat, axis=1, keepdims=True)
        scaled_g = g / (g_norm.reshape((g.shape[0],) + (1,) * (g.ndim - 1)) + 1e-12)
        adv = adv + step_size * scaled_g
        adv = project_l2(adv)
        adv = jnp.clip(adv, 0.0, 1.0)
        return adv

    adv = project_l2(x)
    adv = jax.lax.fori_loop(0, num_steps, step_fn, adv)
    return adv

def evaluate_pgd(
    state: train_state.TrainState,
    images: jnp.ndarray,
    labels: jnp.ndarray,
    batch_size: int,
    eps: float = 0.3,
    step_size: Optional[float] = None,
    num_steps: int = 40,
) -> Dict[str, float]:
    total_acc = 0.0
    total_n = 0
    total_l2 = 0.0
    total_linf = 0.0

    for x, y in eval_iterator(images, labels, batch_size):
        if x.shape[0] == 0:
            continue
        adv_x = pgd_l2_attack(state.apply_fn, state.params, x, y, eps, step_size, num_steps)
        logits = state.apply_fn({"params": state.params}, adv_x)
        acc = accuracy(logits, y)
        n = x.shape[0]
        total_acc += float(acc) * n
        total_n += n
        diff = adv_x - x
        flat = diff.reshape((diff.shape[0], -1))
        total_l2 += float(jnp.linalg.norm(flat, axis=1).mean()) * n
        total_linf += float(jnp.max(jnp.abs(diff), axis=tuple(range(1, diff.ndim))).mean()) * n

    return {
        "acc": total_acc / total_n,
        "avg_l2": total_l2 / total_n,
        "avg_linf": total_linf / total_n,
    }


def load_mnist() -> Tuple[Tuple[jnp.ndarray, jnp.ndarray],
                           Tuple[jnp.ndarray, jnp.ndarray]]:
    """Load MNIST using torchvision (no TensorFlow).

    Shapes:
      train_images: [60000, 28, 28, 1]
      train_labels: [60000]
      test_images:  [10000, 28, 28, 1]
      test_labels:  [10000]
    """
    train_ds = torchvision.datasets.MNIST(root="data", train=True, download=True)
    test_ds = torchvision.datasets.MNIST(root="data", train=False, download=True)

    train_images = jnp.asarray(train_ds.data.numpy(), dtype=jnp.float32) / 255.0
    test_images = jnp.asarray(test_ds.data.numpy(), dtype=jnp.float32) / 255.0
    train_labels = jnp.asarray(train_ds.targets.numpy(), dtype=jnp.int32)
    test_labels = jnp.asarray(test_ds.targets.numpy(), dtype=jnp.int32)

    if train_images.ndim == 3:
        train_images = train_images[..., None]
    if test_images.ndim == 3:
        test_images = test_images[..., None]

    return (train_images, train_labels), (test_images, test_labels)

def data_iterator(
    rng: jax.random.PRNGKey,
    images: jnp.ndarray,
    labels: jnp.ndarray,
    batch_size: int,
) -> Iterable[Tuple[jnp.ndarray, jnp.ndarray]]:
    """Simple shuffled mini-batch iterator with no empty batches."""
    num = images.shape[0]
    perm = jax.random.permutation(rng, num)
    for i in range(0, num, batch_size):
        idx = perm[i:i + batch_size]
        yield images[idx], labels[idx]


def eval_iterator(
    images: jnp.ndarray,
    labels: jnp.ndarray,
    batch_size: int,
) -> Iterable[Tuple[jnp.ndarray, jnp.ndarray]]:
    """Sequential mini-batch iterator for evaluation."""
    num = images.shape[0]
    for i in range(0, num, batch_size):
        yield images[i:i + batch_size], labels[i:i + batch_size]


# ---------------------------------------------------------------------------
#  Classifier: LeNet-like convnet
# ---------------------------------------------------------------------------

class LeNet(nn.Module):
    num_classes: int = 10

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # x: [B, 28, 28, 1]
        x = nn.Conv(features=32, kernel_size=(5, 5), strides=(1, 1))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = nn.Conv(features=64, kernel_size=(5, 5), strides=(1, 1))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=256)(x)
        x = nn.relu(x)
        x = nn.Dense(features=self.num_classes)(x)
        return x


def create_classifier_state(
    rng: jax.random.PRNGKey,
    learning_rate: float,
) -> train_state.TrainState:
    model = LeNet(num_classes=10)
    dummy_x = jnp.zeros((1, 28, 28, 1), dtype=jnp.float32)
    params = model.init(rng, dummy_x)["params"]
    tx = optax.adam(learning_rate)
    apply_fn = jax.jit(model.apply)
    return train_state.TrainState.create(apply_fn=apply_fn, params=params, tx=tx)


# ---------------------------------------------------------------------------
#  Principled ICNN initialization (dense variant)
# ---------------------------------------------------------------------------

def icnn_principled_moments(fan_in: int) -> Tuple[float, float, float, float, float]:
    """Port of your _icnn_principled_moments to scalars."""
    if fan_in <= 0:
        raise ValueError(f"ICNN fan-in must be positive; got {fan_in}.")
    denom_offset = 6.0 * (math.pi - 1.0)
    denom_slope = 3.0 * math.sqrt(3.0) + 2.0 * math.pi - 6.0
    denom = denom_offset + (fan_in - 1.0) * denom_slope
    mu_w = math.sqrt((6.0 * math.pi) / (fan_in * denom))
    sigma_w2 = 1.0 / float(fan_in)
    mu_b = math.sqrt((3.0 * fan_in) / denom)
    mu_w_sq = mu_w * mu_w
    log_var_plus_mean_sq = math.log(sigma_w2 + mu_w_sq)
    log_mean_sq = math.log(mu_w_sq)
    tilde_mu = log_mean_sq - 0.5 * log_var_plus_mean_sq
    tilde_sigma2 = max(log_var_plus_mean_sq - log_mean_sq, 1e-12)
    tilde_sigma = math.sqrt(tilde_sigma2)
    return mu_w, sigma_w2, mu_b, tilde_mu, tilde_sigma


class NonNegativeDense(nn.Module):
    """Linear map with weights constrained to be non-negative (exp or softplus)."""
    features: int
    use_bias: bool = True
    init_mode: str = "principled"  # "principled" or "xavier"

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        in_features = x.shape[-1]
        init_mode = self.init_mode.lower()
        if init_mode not in {"principled", "xavier"}:
            raise ValueError(f"Unsupported init_mode '{self.init_mode}' for NonNegativeDense.")
        parametrisation = "exp" if init_mode == "principled" else "softplus"

        def weight_init(key, shape, dtype=jnp.float32):
            fan_in = shape[0]
            if init_mode == "principled":
                _, _, _, tilde_mu, tilde_sigma = icnn_principled_moments(fan_in)
                if tilde_sigma == 0.0:
                    w = jnp.full(shape, tilde_mu, dtype)
                else:
                    noise = jax.random.normal(key, shape, dtype)
                    w = noise * tilde_sigma + tilde_mu
                return w
            else:
                return nn.initializers.xavier_uniform()(key, shape, dtype)

        weight_param = self.param(
            "weight_param",
            weight_init,
            (in_features, self.features),
        )

        if parametrisation == "exp":
            weight = jnp.exp(weight_param)
        else:
            weight = jax.nn.softplus(weight_param)

        y = jnp.dot(x, weight)  # [B, features]

        if self.use_bias:
            def bias_init(key, shape, dtype=jnp.float32):
                if init_mode == "principled":
                    fan_in = in_features
                    _, _, mu_b, _, _ = icnn_principled_moments(fan_in)
                    return jnp.full(shape, mu_b, dtype)
                else:
                    return jnp.zeros(shape, dtype)

            bias = self.param("bias", bias_init, (self.features,))
            y = y + bias

        return y


class InputConvexPotential(nn.Module):
    """
    Dense FICNN-style potential φ(z), matching your InputConvexPotential's
    non-convolutional branch. Convexity enforced via non-negative hidden
    couplings + quadratic term.
    """
    input_dim: int
    hidden_sizes: Sequence[int]
    activation: str = "relu"
    strong_convexity: float = 1.0
    nonneg_init: str = "principled"

    @nn.compact
    def __call__(self, z: jnp.ndarray) -> jnp.ndarray:
        # z: [B, D] or [B, ...]; flatten spatial dims.
        z_flat = z.reshape((z.shape[0], -1))

        act_name = self.activation.lower()
        if act_name == "relu":
            act = nn.relu
        elif act_name == "softplus":
            act = lambda u: jax.nn.softplus(20.0 * u) / 20.0
        else:
            raise ValueError(f"Unsupported ICNN activation '{self.activation}'.")

        if len(self.hidden_sizes) == 0:
            raise ValueError("ICNN requires at least one hidden layer.")

        h = None
        for i, width in enumerate(self.hidden_sizes):
            # z-linear part: arbitrary signed weights.
            z_linear = nn.Dense(features=width, name=f"z_linear_{i}")
            z_term = z_linear(z_flat)
            if h is None:
                h = act(z_term)
            else:
                # h-linear part: non-negative weights via NonNegativeDense
                h_linear = NonNegativeDense(
                    features=width,
                    init_mode=self.nonneg_init,
                    name=f"h_linear_{i}",
                )
                h = act(z_term + h_linear(h))

        assert h is not None
        hidden_output = NonNegativeDense(
            features=1,
            init_mode=self.nonneg_init,
            name="hidden_output",
        )
        input_skip = nn.Dense(features=1, name="input_skip")

        quadratic = 0.5 * self.strong_convexity * jnp.sum(z_flat ** 2, axis=1, keepdims=True)
        out = quadratic + input_skip(z_flat) + hidden_output(h)
        return jnp.squeeze(out, axis=-1)  # [B]


def icnn_gradient(apply_fn, params, z_flat: jnp.ndarray) -> jnp.ndarray:
    """∇φ(z_flat) where φ is the InputConvexPotential."""
    def phi_sum(u):
        # scalar: sum over batch for convenience
        return apply_fn({"params": params}, u).sum()

    grad_phi = jax.grad(phi_sum)(z_flat)
    return grad_phi  # same shape as z_flat


# ---------------------------------------------------------------------------
#  BB + Armijo line search (for ascent)
# ---------------------------------------------------------------------------

@dataclass
class BBArmijoState:
    alpha_min: float
    alpha_max: float
    alpha_prev: float
    ls_c: float
    ls_shrink: float
    ls_max_steps: int
    prev_params_vec: Optional[jnp.ndarray] = None
    prev_grad_vec: Optional[jnp.ndarray] = None

    @classmethod
    def create(
        cls,
        alpha0: float = 1e-1,
        alpha_min: float = 1e-6,
        alpha_max: float = 10.0,
        ls_c: float = 1e-4,
        ls_shrink: float = 0.5,
        ls_max_steps: int = 10,
    ) -> "BBArmijoState":
        alpha0 = float(max(alpha_min, min(alpha_max, alpha0)))
        return cls(
            alpha_min=float(max(alpha_min, 1e-12)),
            alpha_max=float(max(alpha_max, alpha_min)),
            alpha_prev=alpha0,
            ls_c=float(ls_c),
            ls_shrink=float(ls_shrink),
            ls_max_steps=int(max(ls_max_steps, 1)),
        )

    def propose(self, params_vec: jnp.ndarray, grad_vec: jnp.ndarray) -> float:
        """Propose a BB step size (no Armijo yet)."""
        if (
            self.prev_params_vec is None
            or self.prev_grad_vec is None
            or self.prev_params_vec.shape != params_vec.shape
            or self.prev_grad_vec.shape != grad_vec.shape
        ):
            alpha = self.alpha_prev
        else:
            s = params_vec - self.prev_params_vec
            y = grad_vec - self.prev_grad_vec
            denom = jnp.dot(s, y)
            num = jnp.dot(s, s)
            cond = jnp.isfinite(denom) & (jnp.abs(denom) > 1e-12)
            alpha_bb = jnp.where(cond, num / denom, self.alpha_prev)
            alpha = float(jnp.clip(alpha_bb, self.alpha_min, self.alpha_max))

        if not math.isfinite(alpha):
            alpha = self.alpha_prev
        alpha = max(self.alpha_min, min(self.alpha_max, float(alpha)))
        return alpha

    def update_history(self, params_vec: jnp.ndarray, grad_vec: jnp.ndarray, alpha: float) -> "BBArmijoState":
        alpha_clamped = max(self.alpha_min, min(self.alpha_max, float(alpha)))
        return BBArmijoState(
            alpha_min=self.alpha_min,
            alpha_max=self.alpha_max,
            alpha_prev=alpha_clamped,
            ls_c=self.ls_c,
            ls_shrink=self.ls_shrink,
            ls_max_steps=self.ls_max_steps,
            prev_params_vec=jnp.array(params_vec),
            prev_grad_vec=jnp.array(grad_vec),
        )


def bb_armijo_ascent_x(
    x0: jnp.ndarray,
    f,
    num_steps: int,
    bb_state: Optional[BBArmijoState] = None,
) -> jnp.ndarray:
    """
    Gradient ascent on x with BB step size + Armijo backtracking.

    f: x -> scalar objective (maximized).
    """
    if x0.size == 0 or num_steps == 0:
        return x0

    if bb_state is None:
        bb_state = BBArmijoState.create()
    grad_f = jax.grad(f)

    x = x0
    state = bb_state
    for _ in range(num_steps):
        g = grad_f(x)
        x_vec = x.ravel()
        g_vec = g.ravel()
        alpha = state.propose(x_vec, g_vec)

        fx = float(f(x))
        g_dot_g = float(jnp.dot(g_vec, g_vec))
        if g_dot_g == 0.0:
            break

        alpha_k = alpha
        for _ in range(state.ls_max_steps):
            x_trial = x + alpha_k * g
            f_trial = float(f(x_trial))
            if f_trial >= fx + state.ls_c * alpha_k * g_dot_g:
                break
            alpha_k *= state.ls_shrink

        x_new = x + alpha_k * g
        g_new = grad_f(x_new)
        state = state.update_history(x_new.ravel(), g_new.ravel(), alpha_k)
        x = x_new

    return x


def bb_armijo_step_params(
    params: Any,
    f_params,
    bb_state: BBArmijoState,
) -> Tuple[Any, BBArmijoState, float]:
    """
    Single BB+Armijo gradient-ascent step on a parameter PyTree.

    f_params: params -> scalar objective (maximized).
    """
    params_vec, unravel = ravel_pytree(params)

    def f_vec(v):
        return f_params(unravel(v))

    grad_vec = jax.grad(f_vec)(params_vec)
    alpha = bb_state.propose(params_vec, grad_vec)
    f_val = float(f_vec(params_vec))
    g_dot_g = float(jnp.dot(grad_vec, grad_vec))
    if g_dot_g == 0.0:
        return params, bb_state, f_val

    alpha_k = alpha
    for _ in range(bb_state.ls_max_steps):
        v_trial = params_vec + alpha_k * grad_vec
        f_trial = float(f_vec(v_trial))
        if f_trial >= f_val + bb_state.ls_c * alpha_k * g_dot_g:
            break
        alpha_k *= bb_state.ls_shrink

    v_new = params_vec + alpha_k * grad_vec
    grad_vec_new = jax.grad(f_vec)(v_new)
    new_bb_state = bb_state.update_history(v_new, grad_vec_new, alpha_k)
    new_params = unravel(v_new)
    return new_params, new_bb_state, f_val


# ---------------------------------------------------------------------------
#  Training config and ICNN state
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class TrainConfig:
    batch_size: int = 256
    num_epochs: int = 3
    lr_cls: float = 1e-3
    lambda_reg: float = 0.5
    log_every: int = 100
    max_steps_algo1: Optional[int] = None
    max_steps_algo2: Optional[int] = None
    use_margin_adv_algo1: bool = False
    use_margin_adv_algo2: bool = False
    inner_steps_algo1: int = 5
    inner_steps_algo2: int = 5
    bb_alpha0_x: float = 1e-1
    bb_alpha0_icnn: float = 1e-2
    icnn_hidden_sizes: Sequence[int] = (64, 64, 64, 64)
    seed: int = 0


@dataclass
class ICNNState:
    params: Any
    apply_fn: Any
    bb_state: BBArmijoState


def create_icnn_state(
    rng: jax.random.PRNGKey,
    cfg: TrainConfig,
    input_dim: int,
) -> ICNNState:
    icnn = InputConvexPotential(
        input_dim=input_dim,
        hidden_sizes=cfg.icnn_hidden_sizes,
        activation="softplus",
        strong_convexity=1.0,
        nonneg_init="principled",
    )
    dummy_flat = jnp.zeros((1, input_dim), dtype=jnp.float32)
    params = icnn.init(rng, dummy_flat)["params"]
    bb_state = BBArmijoState.create(alpha0=cfg.bb_alpha0_icnn)
    return ICNNState(params=params, apply_fn=icnn.apply, bb_state=bb_state)


# ---------------------------------------------------------------------------
#  Algorithm 1: GD + particle ascent (with BB+Armijo)
# ---------------------------------------------------------------------------

def train_step_algo1(
    state: train_state.TrainState,
    batch: Tuple[jnp.ndarray, jnp.ndarray],
    cfg: TrainConfig,
) -> Tuple[train_state.TrainState, Dict[str, jnp.ndarray]]:
    x, y = batch  # x: [B,28,28,1], y: [B]

    if x.shape[0] == 0:
        metrics = {
            "loss_adv": jnp.array(0.0),
            "acc_clean": jnp.array(0.0),
            "acc_adv": jnp.array(0.0),
            "w2_proxy": jnp.array(0.0),
        }
        return state, metrics

    def adv_obj(z: jnp.ndarray) -> jnp.ndarray:
        logits = state.apply_fn({"params": state.params}, z)
        adv_loss = adversary_loss(logits, y, cfg.use_margin_adv_algo1)
        sq_dist = jnp.mean(jnp.sum((z - x) ** 2, axis=(1, 2, 3)))
        return adv_loss - cfg.lambda_reg * sq_dist

    adv_x = bb_armijo_ascent_x(
        x0=x,
        f=adv_obj,
        num_steps=cfg.inner_steps_algo1,
        bb_state=BBArmijoState.create(alpha0=cfg.bb_alpha0_x),
    )
    adv_x = jax.lax.stop_gradient(adv_x)

    def loss_fn(params):
        logits = state.apply_fn({"params": params}, adv_x)
        return cross_entropy_loss(logits, y).mean()

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    new_state = state.apply_gradients(grads=grads)

    clean_logits = new_state.apply_fn({"params": new_state.params}, x)
    adv_logits = new_state.apply_fn({"params": new_state.params}, adv_x)

    metrics = {
        "loss_adv": loss,
        "acc_clean": accuracy(clean_logits, y),
        "acc_adv": accuracy(adv_logits, y),
        "w2_proxy": jnp.mean(jnp.sum((adv_x - x) ** 2, axis=(1, 2, 3))),
    }
    return new_state, metrics



def evaluate_clean(
    state: train_state.TrainState,
    images: jnp.ndarray,
    labels: jnp.ndarray,
    batch_size: int,
) -> Dict[str, float]:
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    def eval_step(params, x, y):
        logits = state.apply_fn({"params": params}, x)
        loss = cross_entropy_loss(logits, y).mean()
        acc = accuracy(logits, y)
        return loss, acc

    eval_step_jit = jax.jit(eval_step)

    for x, y in eval_iterator(images, labels, batch_size):
        if x.shape[0] == 0:
            continue
        loss, acc = eval_step_jit(state.params, x, y)
        n = x.shape[0]
        total_loss += float(loss) * n
        total_acc += float(acc) * n
        total_n += n
    return {"loss": total_loss / total_n, "acc": total_acc / total_n}


def train_algorithm_1(cfg: TrainConfig) -> Tuple[train_state.TrainState, Dict[str, Any]]:
    rng = jax.random.PRNGKey(cfg.seed)
    (train_images, train_labels), (test_images, test_labels) = load_mnist()
    train_images = jax.device_put(train_images)
    train_labels = jax.device_put(train_labels)
    test_images = jax.device_put(test_images)
    test_labels = jax.device_put(test_labels)

    state = create_classifier_state(rng, cfg.lr_cls)
    logger = CSVLogger(DEFAULT_LOG_PATH, LOG_FIELDNAMES)

    for epoch in range(cfg.num_epochs):
        rng, epoch_key = jax.random.split(rng)
        for step, (x, y) in enumerate(
            data_iterator(epoch_key, train_images, train_labels, cfg.batch_size)
        ):
            if cfg.max_steps_algo1 is not None and step >= cfg.max_steps_algo1:
                break

            state, metrics = train_step_algo1(state, (x, y), cfg)
            if cfg.log_every and step % cfg.log_every == 0:
                print(
                    f"[Algo1] Epoch {epoch} step {step} "
                    f"loss_adv={float(metrics['loss_adv']):.4f} "
                    f"acc_clean={float(metrics['acc_clean']):.4f} "
                    f"acc_adv={float(metrics['acc_adv']):.4f} "
                    f"W2≈{float(metrics['w2_proxy']):.4f}"
                )
                logger.log(
                    algorithm="algo1",
                    phase="train",
                    epoch=epoch,
                    step=step,
                    loss_adv=float(metrics['loss_adv']),
                    adv_loss=None,
                    cls_loss=None,
                    acc_clean=float(metrics['acc_clean']),
                    acc_adv=float(metrics['acc_adv']),
                    w2_proxy=float(metrics['w2_proxy']),
                )

    test_metrics = evaluate_clean(state, test_images, test_labels, cfg.batch_size)
    print("[Algo1] Test:", test_metrics)
    logger.log(
        algorithm="algo1",
        phase="test",
        epoch=cfg.num_epochs,
        step=None,
        loss_adv=float(test_metrics["loss"]),
        adv_loss=None,
        cls_loss=None,
        acc_clean=float(test_metrics["acc"]),
        acc_adv=None,
        w2_proxy=None,
    )
    return state, {"test": test_metrics}


# ---------------------------------------------------------------------------
#  Algorithm 2: ICNN-based adversarial transport (BB+Armijo on ICNN params)
# ---------------------------------------------------------------------------

def train_step_algo2(
    state: train_state.TrainState,
    icnn_state: ICNNState,
    batch: Tuple[jnp.ndarray, jnp.ndarray],
    cfg: TrainConfig,
) -> Tuple[train_state.TrainState, ICNNState, Dict[str, jnp.ndarray]]:
    x, y = batch
    if x.shape[0] == 0:
        metrics = {
            "adv_loss": jnp.array(0.0),
            "cls_loss": jnp.array(0.0),
            "acc_clean": jnp.array(0.0),
            "acc_adv": jnp.array(0.0),
            "w2_proxy": jnp.array(0.0),
        }
        return state, icnn_state, metrics

    # Freeze classifier params during ICNN inner maximization
    theta_eval = jax.lax.stop_gradient(state.params)
    x_flat = x.reshape((x.shape[0], -1))

    # Inner maximization over ICNN params (BB+Armijo).
    def adv_obj_params(params) -> jnp.ndarray:
        adv_flat = icnn_gradient(icnn_state.apply_fn, params, x_flat)
        adv_x = adv_flat.reshape(x.shape)
        logits = state.apply_fn({"params": theta_eval}, adv_x, train=False)
        adv_loss = adversary_loss(logits, y, cfg.use_margin_adv_algo2)
        w2 = jnp.mean(jnp.sum((adv_flat - x_flat) ** 2, axis=1))
        return adv_loss - cfg.lambda_reg * w2

    icnn_params = icnn_state.params
    bb_state = icnn_state.bb_state
    adv_loss_val = jnp.array(0.0)
    for _ in range(cfg.inner_steps_algo2):
        icnn_params, bb_state, adv_loss_val = bb_armijo_step_params(
            params=icnn_params,
            f_params=adv_obj_params,
            bb_state=bb_state,
        )

    icnn_state = ICNNState(
        params=icnn_params,
        apply_fn=icnn_state.apply_fn,
        bb_state=bb_state,
    )

    # Transport with updated ICNN to update classifier.
    adv_flat = jax.lax.stop_gradient(
        icnn_gradient(icnn_state.apply_fn, icnn_state.params, x_flat)
    )
    adv_x = adv_flat.reshape(x.shape)
    adv_x_stop = adv_x

    def cls_loss(params):
        logits = state.apply_fn({"params": params}, adv_x_stop)
        return cross_entropy_loss(logits, y).mean()

    cls_loss_val, grads_theta = jax.value_and_grad(cls_loss)(theta_eval)
    new_state = state.apply_gradients(grads=grads_theta)

    clean_logits = new_state.apply_fn({"params": new_state.params}, x)
    adv_logits = new_state.apply_fn({"params": new_state.params}, adv_x_stop)
    w2_val = jnp.mean(jnp.sum((adv_flat - x_flat) ** 2, axis=1))

    metrics = {
        "adv_loss": jnp.array(adv_loss_val),
        "cls_loss": cls_loss_val,
        "acc_clean": accuracy(clean_logits, y),
        "acc_adv": accuracy(adv_logits, y),
        "w2_proxy": w2_val,
    }
    return new_state, icnn_state, metrics



def train_algorithm_2(cfg: TrainConfig) -> Tuple[train_state.TrainState, ICNNState, Dict[str, Any]]:
    rng = jax.random.PRNGKey(cfg.seed)
    (train_images, train_labels), (test_images, test_labels) = load_mnist()
    train_images = jax.device_put(train_images)
    train_labels = jax.device_put(train_labels)
    test_images = jax.device_put(test_images)
    test_labels = jax.device_put(test_labels)

    state = create_classifier_state(rng, cfg.lr_cls)
    logger = CSVLogger(DEFAULT_LOG_PATH, LOG_FIELDNAMES)

    input_dim = 28 * 28 * 1
    rng, icnn_rng = jax.random.split(rng)
    icnn_state = create_icnn_state(icnn_rng, cfg, input_dim)

    for epoch in range(cfg.num_epochs):
        rng, epoch_key = jax.random.split(rng)
        for step, (x, y) in enumerate(
            data_iterator(epoch_key, train_images, train_labels, cfg.batch_size)
        ):
            if cfg.max_steps_algo2 is not None and step >= cfg.max_steps_algo2:
                break

            state, icnn_state, metrics = train_step_algo2(state, icnn_state, (x, y), cfg)
            if cfg.log_every and step % cfg.log_every == 0:
                print(
                    f"[Algo2] Epoch {epoch} step {step} "
                    f"adv_loss={float(metrics['adv_loss']):.4f} "
                    f"cls_loss={float(metrics['cls_loss']):.4f} "
                    f"acc_clean={float(metrics['acc_clean']):.4f} "
                    f"acc_adv={float(metrics['acc_adv']):.4f} "
                    f"W2≈{float(metrics['w2_proxy']):.4f}"
                )
                logger.log(
                    algorithm="algo2",
                    phase="train",
                    epoch=epoch,
                    step=step,
                    loss_adv=None,
                    adv_loss=float(metrics['adv_loss']),
                    cls_loss=float(metrics['cls_loss']),
                    acc_clean=float(metrics['acc_clean']),
                    acc_adv=float(metrics['acc_adv']),
                    w2_proxy=float(metrics['w2_proxy']),
                )

    test_metrics = evaluate_clean(state, test_images, test_labels, cfg.batch_size)
    print("[Algo2] Test:", test_metrics)
    logger.log(
        algorithm="algo2",
        phase="test",
        epoch=cfg.num_epochs,
        step=None,
        loss_adv=float(test_metrics["loss"]),
        adv_loss=None,
        cls_loss=float(test_metrics["loss"]),
        acc_clean=float(test_metrics["acc"]),
        acc_adv=None,
        w2_proxy=None,
    )
    return state, icnn_state, {"test": test_metrics}


# ---------------------------------------------------------------------------
#  Main
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    cfg = TrainConfig(
        batch_size=512,
        num_epochs=10,         # increase for full experiments
        lr_cls=1e-3,
        lambda_reg=0.5,       
        log_every=100,
        max_steps_algo1=None,
        max_steps_algo2=None,
        use_margin_adv_algo1=False,
        use_margin_adv_algo2=False,
        inner_steps_algo1=5,
        inner_steps_algo2=5,
        bb_alpha0_x=1e-1,
        bb_alpha0_icnn=1e-2,
        icnn_hidden_sizes=(512, 512, 256, 128),
        seed=0,
    )

    print("Training Algorithm 1 (particle ascent with BB+Armijo)...")
    state_algo1, logs_algo1 = train_algorithm_1(cfg)

    print("\nTraining Algorithm 2 (ICNN transport with BB+Armijo)...")
    state_algo2, icnn_state_algo2, logs_algo2 = train_algorithm_2(cfg)
    # PGD robustness evaluation for both algorithms
    (_, _), (test_images, test_labels) = load_mnist()
    pgd_kwargs = dict(batch_size=cfg.batch_size, eps=2.0, num_steps=20)

    pgd_algo1 = evaluate_pgd(state_algo1, test_images, test_labels, **pgd_kwargs)
    print(f"[Algo1] PGD acc={pgd_algo1['acc']*100:.2f}% L2={pgd_algo1['avg_l2']:.4f} Linf={pgd_algo1['avg_linf']:.4f}")

    pgd_algo2 = evaluate_pgd(state_algo2, test_images, test_labels, **pgd_kwargs)
    print(f"[Algo2] PGD acc={pgd_algo2['acc']*100:.2f}% L2={pgd_algo2['avg_l2']:.4f} Linf={pgd_algo2['avg_linf']:.4f}")


Training Algorithm 1 (particle ascent with BB+Armijo)...
[Algo1] Epoch 0 step 0 loss_adv=2.3335 acc_clean=0.3496 acc_adv=0.3477 W2≈0.0000
[Algo1] Epoch 0 step 100 loss_adv=0.1176 acc_clean=0.9609 acc_adv=0.9609 W2≈0.0000
[Algo1] Epoch 1 step 0 loss_adv=0.0516 acc_clean=0.9844 acc_adv=0.9844 W2≈0.0000
[Algo1] Epoch 1 step 100 loss_adv=0.0610 acc_clean=0.9922 acc_adv=0.9922 W2≈0.0000
[Algo1] Epoch 2 step 0 loss_adv=0.0388 acc_clean=0.9883 acc_adv=0.9883 W2≈0.0000
[Algo1] Epoch 2 step 100 loss_adv=0.0299 acc_clean=0.9922 acc_adv=0.9922 W2≈0.0000
[Algo1] Epoch 3 step 0 loss_adv=0.0265 acc_clean=0.9961 acc_adv=0.9961 W2≈0.0000
[Algo1] Epoch 3 step 100 loss_adv=0.0185 acc_clean=0.9961 acc_adv=0.9961 W2≈0.0000
[Algo1] Epoch 4 step 0 loss_adv=0.0356 acc_clean=0.9902 acc_adv=0.9902 W2≈0.0000
[Algo1] Epoch 4 step 100 loss_adv=0.0273 acc_clean=0.9902 acc_adv=0.9902 W2≈0.0000
[Algo1] Epoch 5 step 0 loss_adv=0.0152 acc_clean=0.9961 acc_adv=0.9961 W2≈0.0000
[Algo1] Epoch 5 step 100 loss_adv=0.0184 a

KeyboardInterrupt: 